# Proyecto - Regresión
### Camila Johana González Acosta 599303

**Objetivo:** Predecir `FTResult` (H/A/D) sin usar columnas de goles ni de apuestas.  

## 1) Cargar librerias y datos 

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("MatchesOriginal.csv", low_memory=False)

df.head()


,Division,MatchDate,MatchTime,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,...,MaxUnder25,HandiSize,HandiHome,HandiAway,C_LTH,C_LTA,C_VHD,C_VAD,C_HTB,C_PHB
0,F1,2000-07-28,NaN,Marseille,Troyes,1686.34,1586.57,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,F1,2000-07-28,NaN,Paris SG,Strasbourg,1714.89,1642.51,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,F2,2000-07-28,NaN,Wasquehal,Nancy,1465.08,1633.80,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,F1,2000-07-29,NaN,Auxerre,Sedan,1635.58,1624.22,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,F1,2000-07-29,NaN,Bordeaux,Metz,1734.34,1673.11,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2) Exploración inicial de los datos
A continuación se revisan:
- Dimensiones del dataset  
- Tipos de variables  
- Valores únicos de FTResult  
- Estadísticas descriptivas de las variables predictoras

El objetivo **FTResult** es una variable categórica ordinal no numérica (H, A, D), por lo tanto utilizaremos **regresión logística multinomial**. Además se probará un método no lineal para comparar.


In [29]:
print("Dimensiones:", df.shape)
print("\nTipos:")
print(df.dtypes)

print("\nValores únicos en FTResult:", df["FTResult"].unique())

# Seleccionar solo variables permitidas
vars_model = [
    "HomeFouls", "AwayFouls",
    "HomeCorners", "AwayCorners",
    "HomeYellow", "AwayYellow",
    "HomeRed", "AwayRed"
]

df_sub = df[vars_model + ["FTResult"]]

df_sub.describe()


Dimensiones: (230557, 48)

Tipos:
Division        object
MatchDate       object
MatchTime       object
HomeTeam        object
AwayTeam        object
HomeElo        float64
AwayElo        float64
Form3Home      float64
Form5Home      float64
Form3Away      float64
Form5Away      float64
FTHome         float64
FTAway         float64
FTResult        object
HTHome         float64
HTAway         float64
HTResult        object
HomeShots      float64
AwayShots      float64
HomeTarget     float64
AwayTarget     float64
HomeFouls      float64
AwayFouls      float64
HomeCorners    float64
AwayCorners    float64
HomeYellow     float64
AwayYellow     float64
HomeRed        float64
AwayRed        float64
OddHome        float64
OddDraw        float64
OddAway        float64
MaxHome        float64
MaxDraw        float64
MaxAway        float64
Over25         float64
Under25        float64
MaxOver25      float64
MaxUnder25     float64
HandiSize      float64
HandiHome      float64
HandiAway      float64


,HomeFouls,AwayFouls,HomeCorners,AwayCorners,HomeYellow,AwayYellow,HomeRed,AwayRed
count,113973.000000,113973.000000,114363.000000,114363.000000,119298.000000,119299.000000,119299.000000,119297.00000
mean,12.604231,13.062594,5.656593,4.611526,1.683884,1.985800,0.086388,0.11686
std,4.468416,4.540744,2.940007,2.623002,1.307300,1.374269,0.297656,0.34610
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,9.000000,10.000000,4.000000,3.000000,1.000000,1.000000,0.000000,0.00000
50%,12.000000,13.000000,5.000000,4.000000,2.000000,2.000000,0.000000,0.00000
75%,15.000000,16.000000,7.000000,6.000000,2.000000,3.000000,0.000000,0.00000
max,145.000000,77.000000,26.000000,23.000000,11.000000,10.000000,3.000000,9.00000


## 3) Manejo de problemas típicos C1.5

- **Valores faltantes**: Se revisan y se sustituyen por la mediana.

- **Escalamiento de variables**: Se usa StandardScaler porque los modelos de regresión logística son sensibles a la escala.

- **Codificación de variables categóricas**: FTResult se convierte a valores numéricos con map:
    - H → 0  
    - D → 1  
    - A → 2  

In [30]:
print("Valores faltantes antes:")
print(df_sub.isna().sum())

# Llenar faltantes correctamente usando .loc y evitando warnings
df_sub.loc[:, vars_model[:-1]] = df_sub[vars_model[:-1]].fillna(
    df_sub[vars_model[:-1]].median()
)

print("\nValores faltantes después:")
print(df_sub.isna().sum())

# Codificar el objetivo correctamente
df_sub.loc[:, "FTResult_num"] = df_sub["FTResult"].map({"H": 0, "D": 1, "A": 2})

# Matriz X/y
X = df_sub[vars_model[:-1]]   # todas las variables EXCEPTO FTResult
y = df_sub["FTResult_num"]

# Escalamiento (opcional pero correcto para regresión/logística)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


Valores faltantes antes:
HomeFouls      116584
AwayFouls      116584
HomeCorners    116194
AwayCorners    116194
HomeYellow     111259
AwayYellow     111258
HomeRed        111258
AwayRed        111260
FTResult            3
dtype: int64

Valores faltantes después:
HomeFouls           0
AwayFouls           0
HomeCorners         0
AwayCorners         0
HomeYellow          0
AwayYellow          0
HomeRed             0
AwayRed        111260
FTResult            3
dtype: int64


C:\Users\gonza\AppData\Local\Temp\ipykernel_2080\3820955265.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub.loc[:, "FTResult_num"] = df_sub["FTResult"].map({"H": 0, "D": 1, "A": 2})


## 4) Selección de características

Se utiliza **RFE (Recursive Feature Elimination)** con un modelo base de regresión logística.

Esto permite identificar cuáles variables aportan más información para la predicción.

In [31]:
# Eliminar filas donde FTResult esté vacío
df_sub = df_sub[df_sub["FTResult"].isin(["H", "D", "A"])].copy()

# Manejo de valores faltantes SOLO para variables numéricas
df_sub.loc[:, vars_model[:-1]] = df_sub[vars_model[:-1]].fillna(
    df_sub[vars_model[:-1]].median()
)

# Codificar FTResult correctamente
df_sub["FTResult_num"] = df_sub["FTResult"].map({"H": 0, "D": 1, "A": 2})

# Verificar que ya no existan NaN
print("Valores faltantes en FTResult_num:", df_sub["FTResult_num"].isna().sum())

# Construir X e y
X = df_sub[vars_model[:-1]]
y = df_sub["FTResult_num"]

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Tamaño de X:", X_scaled.shape)
print("Tamaño de y:", y.shape)

logreg_base = LogisticRegression(max_iter=200)
selector = RFE(logreg_base, n_features_to_select=5)

selector.fit(X_scaled, y)

selected_mask = selector.support_
selected_features = np.array(vars_model[:-1])[selected_mask]

print("Variables originales:")
print(vars_model[:-1])

print("\nVariables seleccionadas por RFE:")
print(selected_features)


Valores faltantes en FTResult_num: 0
Tamaño de X: (230554, 7)
Tamaño de y: (230554,)
Variables originales:
['HomeFouls', 'AwayFouls', 'HomeCorners', 'AwayCorners', 'HomeYellow', 'AwayYellow', 'HomeRed']

Variables seleccionadas por RFE:
['HomeFouls' 'HomeCorners' 'HomeYellow' 'AwayYellow' 'HomeRed']


In [32]:
from sklearn.model_selection import cross_val_score

# Cross-validation con Logistic Regression
cv_log = cross_val_score(model_log, X_scaled, y, cv=5)
print("LogReg CV mean:", cv_log.mean())
print("LogReg CV scores:", cv_log)

# Cross-validation con Random Forest
cv_rf = cross_val_score(model_rf, X_scaled, y, cv=5)
print("Random Forest CV mean:", cv_rf.mean())
print("Random Forest CV scores:", cv_rf)


LogReg CV mean: 0.454509575375622
LogReg CV scores: [0.45360109 0.45355772 0.45321073 0.45702761 0.45515073]
Random Forest CV mean: 0.42367946419693575
Random Forest CV scores: [0.4269697  0.42341307 0.43195767 0.41718896 0.41886792]


# 5) Modelos lineal y no lineal

Se entrenarán dos modelos:

**Modelo lineal:** Regresión logística multinomial

**Modelo no lineal:** Random Forest Classifier


In [33]:
# Train-test split sin fuga de datos
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Modelo lineal
model_log = LogisticRegression(max_iter=500)
model_log.fit(X_train, y_train)

# Modelo no lineal
model_rf = RandomForestClassifier(n_estimators=200, random_state=42)
model_rf.fit(X_train, y_train)

# Mostrar coeficientes sin error
coef_df = pd.DataFrame(model_log.coef_, columns=vars_model[:-1])
print("Coeficientes modelo lineal (LogReg):")
display(coef_df)


Coeficientes modelo lineal (LogReg):


,HomeFouls,AwayFouls,HomeCorners,AwayCorners,HomeYellow,AwayYellow,HomeRed
0,0.023663,-0.021278,-0.037516,-0.004462,-0.077442,0.035148,-0.112680
1,0.005909,0.026781,0.034336,0.012870,0.021172,0.015683,0.003469
2,-0.029572,-0.005503,0.003180,-0.008408,0.056271,-0.050830,0.109211


# 6) Métricas del modelo

- **Accuracy:** Indica qué porcentaje de predicciones fueron correctas.

- **Matriz de confusión:** Permite observar cuándo el modelo confunde H, D y A.

In [34]:
# --- Logistic Regression ---
pred_log = model_log.predict(X_test)
acc_log = accuracy_score(y_test, pred_log)

# --- Random Forest ---
pred_rf = model_rf.predict(X_test)
acc_rf = accuracy_score(y_test, pred_rf)

print("Accuracy Logistic Regression:", acc_log)
print("Accuracy Random Forest:", acc_rf)

print("\nMatriz de confusión (LogReg):")
print(confusion_matrix(y_test, pred_log))

print("\nClasification report:")
print(classification_report(y_test, pred_log))


Accuracy Logistic Regression: 0.45436013098826744
Accuracy Random Forest: 0.42330463446899874

Matriz de confusión (LogReg):
[[20005    12   558]
 [11612    12   600]
 [12367    11   934]]

Clasification report:
              precision    recall  f1-score   support

           0       0.45      0.97      0.62     20575
           1       0.34      0.00      0.00     12224
           2       0.45      0.07      0.12     13312

    accuracy                           0.45     46111
   macro avg       0.41      0.35      0.25     46111
weighted avg       0.42      0.45      0.31     46111



# 7) Inferencias

A partir del modelo estadístico se pueden identificar:

- Relaciones entre número de faltas, tarjetas y córners con el resultado final.
- Intervalos de confianza para cada coeficiente.
- Significancia estadística (p-values).

Los márgenes de error permiten asegurar si la relación es confiable o podría ser atribuida al azar.

In [35]:
import statsmodels.api as sm

X_inf = sm.add_constant(X_scaled)
model_inf = sm.MNLogit(y, X_inf)
result = model_inf.fit()

result.summary()


Optimization terminated successfully.
         Current function value: 1.064040
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:           FTResult_num   No. Observations:               230554
Model:                        MNLogit   Df Residuals:                   230538
Method:                           MLE   Df Model:                           14
Date:                Sat, 22 Nov 2025   Pseudo R-squ.:                0.006229
Time:                        21:37:40   Log-Likelihood:            -2.4532e+05
converged:                       True   LL-Null:                   -2.4686e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
==================================================================================
FTResult_num=1       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.5145      0.005   -100.253      0.000      -0.525      -0.504
x1                -0.0194      0.006     -3.408      0.001      -0.031      -0.008
x2                 0.0464      0.006      8.114      0.000       0.035       0.058
x3                 0.0728      0.005     13.966      0.000       0.063       0.083
x4                 0.0196      0.005      3.745      0.000       0.009       0.030
x5                 0.1014      0.006     18.057      0.000       0.090       0.112
x6                -0.0192      0.006     -3.441      0.001      -0.030      -0.008
x7                 0.1187      0.006     20.091      0.000       0.107       0.130
----------------------------------------------------------------------------------
FTResult_num=2       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.4364      0.005    -86.961      0.000      -0.446      -0.427
x1                -0.0553      0.006     -9.793      0.000      -0.066      -0.044
x2                 0.0126      0.006      2.224      0.026       0.001       0.024
x3                 0.0414      0.005      8.026      0.000       0.031       0.052
x4                -0.0043      0.005     -0.844      0.399      -0.014       0.006
x5                 0.1345      0.005     24.479      0.000       0.124       0.145
x6                -0.0868      0.006    -15.633      0.000      -0.098      -0.076
x7                 0.2215      0.005     41.573      0.000       0.211       0.232
==================================================================================
"""

Ambos modelos cuentan con un accuracy de menos del 50%, sin embargo es importante considerar que muchos datos fueron perdidos y se esta buscando una conexión entre tarjetas y tiros de esquina para obtener el resultado de un partido. Considerando esto, al tener una base más completa se podría mejorar la precisión del modelo. 